To answer the questions:

1. Where do embeddings succeed but lexical methods fail?
2. Where does TF-IDF succeed but embeddings fail?
3. Which ATT&CK techniques are easiest/hardest to retrieve?
4. How badly do retrieval methods fail when they're wrong?
5. What linguistic characteristics might explain the differences?

In [1]:
#load all 3 query resfult files
import pandas as pd

bow = pd.read_csv("bow_query_results.csv")
tfidf = pd.read_csv("tfidf_query_results.csv")
embedding = pd.read_csv("embedding_query_results.csv")

In [2]:
print("BoW:", bow.shape)
print("TF-IDF:", tfidf.shape)
print("Embedding:", embedding.shape)

BoW: (3077, 4)
TF-IDF: (3077, 4)
Embedding: (3077, 4)


In [6]:
#inspect
bow.head()

,sentence,technique_id,bow_rank,split
0,It spreads to Microsoft Windows machines using...,T1210,29,train
1,"Once a machine is infected by NotPetya, a seri...",T1570,52,train
2,Ransomware DLL C:\windows\perfc.dat,T1140,122,train
3,The malware decompresses its resource named 0x...,T1140,42,train
4,The malware is a DLL that is launched using ru...,T1218.011,1,train


In [4]:
tfidf.head()

,sentence,technique_id,tfidf_rank,split
0,It spreads to Microsoft Windows machines using...,T1210,10,train
1,"Once a machine is infected by NotPetya, a seri...",T1570,31,train
2,Ransomware DLL C:\windows\perfc.dat,T1140,122,train
3,The malware decompresses its resource named 0x...,T1140,22,train
4,The malware is a DLL that is launched using ru...,T1218.011,1,train


In [5]:
embedding.head() 

,sentence,technique_id,embedding_rank,split
0,It spreads to Microsoft Windows machines using...,T1210,24,train
1,"Once a machine is infected by NotPetya, a seri...",T1570,20,train
2,Ransomware DLL C:\windows\perfc.dat,T1140,27,train
3,The malware decompresses its resource named 0x...,T1140,14,train
4,The malware is a DLL that is launched using ru...,T1218.011,2,train


In [8]:
#merge all of them
results = bow[
    [
        "sentence",
        "technique_id",
        "split",
        "bow_rank"
    ]
].copy()

results["tfidf_rank"] = tfidf["tfidf_rank"]

results["embedding_rank"] = embedding["embedding_rank"]

In [9]:
results.head()

,sentence,technique_id,split,bow_rank,tfidf_rank,embedding_rank
0,It spreads to Microsoft Windows machines using...,T1210,train,29,10,24
1,"Once a machine is infected by NotPetya, a seri...",T1570,train,52,31,20
2,Ransomware DLL C:\windows\perfc.dat,T1140,train,122,122,27
3,The malware decompresses its resource named 0x...,T1140,train,42,22,14
4,The malware is a DLL that is launched using ru...,T1218.011,train,1,1,2


In [10]:
#results
mitre = pd.read_csv(
    "mitre_techniques.csv"
)

In [11]:
#add technique names
technique_names = (
    mitre
    .set_index("technique_id")["name"]
    .to_dict()
)

In [12]:
results["technique_name"] = (
    results["technique_id"]
    .map(technique_names)
)

In [13]:
results[
    [
        "sentence",
        "technique_id",
        "technique_name",
        "bow_rank",
        "tfidf_rank",
        "embedding_rank"
    ]
].head()

,sentence,technique_id,technique_name,bow_rank,tfidf_rank,embedding_rank
0,It spreads to Microsoft Windows machines using...,T1210,Exploitation of Remote Services,29,10,24
1,"Once a machine is infected by NotPetya, a seri...",T1570,Lateral Tool Transfer,52,31,20
2,Ransomware DLL C:\windows\perfc.dat,T1140,Deobfuscate/Decode Files or Information,122,122,27
3,The malware decompresses its resource named 0x...,T1140,Deobfuscate/Decode Files or Information,42,22,14
4,The malware is a DLL that is launched using ru...,T1218.011,Rundll32,1,1,2


- in the furutre, the project will probably use top 5 results, so creating top 5 rank

In [14]:
#add success/failure
#top 5 rank
results["bow_top5"] = (
    results["bow_rank"] <= 5
)

results["tfidf_top5"] = (
    results["tfidf_rank"] <= 5
)

results["embedding_top5"] = (
    results["embedding_rank"] <= 5
)

In [15]:
#also top 1
results["bow_top1"] = (
    results["bow_rank"] == 1
)

results["tfidf_top1"] = (
    results["tfidf_rank"] == 1
)

results["embedding_top1"] = (
    results["embedding_rank"] == 1
)

- investigating where embeddings succeeded but TF-IDF failed

In [16]:
embedding_wins = results[
    (results["embedding_top5"])
    &
    (~results["tfidf_top5"])
].copy()

In [17]:
print(
    "Embedding Top-5 successes where TF-IDF failed:",
    len(embedding_wins)
)

Embedding Top-5 successes where TF-IDF failed: 474


In [19]:
#compare an example
pd.set_option(
    "display.max_colwidth",
    None
)

embedding_wins[
    [
        "sentence",
        "technique_id",
        "technique_name",
        "tfidf_rank",
        "embedding_rank"
    ]
].head(10)

,sentence,technique_id,technique_name,tfidf_rank,embedding_rank
17,This is used as an integrity check for the decryption process.,T1140,Deobfuscate/Decode Files or Information,31,2
30,Explanation of schtask parameters: /TN → taskname (not providing a custom name forces Windows to provide a random GUID as a name) /TR → taskrun.,T1053.005,Scheduled Task,47,1
42,PowerShell script to create a ImmediateTask through GPO,T1053.005,Scheduled Task,34,1
43,"As shown in Figure 6, threat actors use multi-layered AES encryption and base64 encoding to obfuscate “co.ps1”.",T1027,Obfuscated Files or Information,54,4
44,Heavy obfuscation in a simple but useful anti-analysis approach makes it difficult for security products to detect their scripts.,T1027,Obfuscated Files or Information,16,2
45,Heavily obfuscated PowerShell script Figure 7.,T1027,Obfuscated Files or Information,25,4
52,"Furthermore, the configuration file will be deleted once it is loaded into memory for fear of leaking network infrastructures.",T1070.004,File Deletion,197,2
55,EarthWorm is a simple network tunnel tool with SOCKS v5 server and port transfer developed by a Chinese engineer.,T1090,Proxy,32,3
56,"In order to avoid being detected by security products, threat actors use Themida packer to obfuscate the signature used for detection.",T1027,Obfuscated Files or Information,11,5
61,"In addition, we reviewed reports published around that time and found the decryption algorithm in ShellFang was once used in the campaign by 1937CN, which was revealed by Fortinet in 2017.",T1140,Deobfuscate/Decode Files or Information,17,1


- this shows evidence supporting semantic-retrieval hypothesis.

**Calculate how much embeddings improved the rank**

In [20]:
results["embedding_vs_tfidf"] = (
    results["tfidf_rank"]
    - results["embedding_rank"]
)

In [24]:
biggest_embedding_wins = (
    results
    .sort_values(
        "embedding_vs_tfidf",
        ascending=False
    )
)

biggest_embedding_wins[
    [
        "sentence",
        "technique_id",
        "technique_name",
        "tfidf_rank",
        "embedding_rank",
        "embedding_vs_tfidf"
    ]
].head()

,sentence,technique_id,technique_name,tfidf_rank,embedding_rank,embedding_vs_tfidf
1398,"After the strings are decrypted, the structure referencing them is never used again.",T1140,Deobfuscate/Decode Files or Information,685,18,667
592,The collected information is then compressed and exfiltrated to the attacker via Telegram.,T1041,Exfiltration Over C2 Channel,668,4,664
408,"In this attack, we observed the adversary attempting to exfiltrate sensitive information over SMB (TCP/445) directly from a compromised domain controller.",T1041,Exfiltration Over C2 Channel,669,11,658
1784,The -e is short for -EncodedCommand.,T1027,Obfuscated Files or Information,667,10,657
2546,That is why malware hidden inside the Python compiled byte code could slip under the radar of the traditional security solutions.,T1027,Obfuscated Files or Information,661,5,656


**where does TF-IDF beat embeddings?**

In [25]:
tfidf_wins = results[
    (results["tfidf_top5"])
    &
    (~results["embedding_top5"])
].copy()

In [26]:
print(
    "TF-IDF Top-5 successes where embeddings failed:",
    len(tfidf_wins)
)

TF-IDF Top-5 successes where embeddings failed: 305


In [27]:
tfidf_wins[
    [
        "sentence",
        "technique_id",
        "technique_name",
        "tfidf_rank",
        "embedding_rank"
    ]
].head(10)

,sentence,technique_id,technique_name,tfidf_rank,embedding_rank
11,This thread is then used to execute the SMB copy and remote execution section as the impersonated user.,T1570,Lateral Tool Transfer,5,196
33,This is an undocumented Windows API that causes the system to reboot.,T1106,Native API,2,20
37,The malware also spawns cmd.exe to execute the following command:,T1059.003,Windows Command Shell,1,10
72,"Strings have been searched, but since there is too much obfuscated information, no tangible results have been found.",T1027,Obfuscated Files or Information,4,10
73,"The stealer also targets crypto wallets such as Binance, Electrum, and Ethereum and collects sensitive information from the victim’s machine.",T1005,Data from Local System,3,148
96,"After getting the configuration file, the malware will parse it and save the fields in memory and write in the registry information to encrypt the files in the machine.",T1112,Modify Registry,3,23
97,"The malware will try first to write in the registry-hive “HKEY_LOCAL_MACHINE” but if it cannot create it, it will use the registry-hive “HKEY_CURRENT_USER”: Figure 12.",T1112,Modify Registry,3,21
118,"However, it’s a link (shortcut) file, as seen in Figure 4.",T1204.002,Malicious File,5,13
121,"This behavior led us to believe that the threat actor behind this attack relies on templates and uses them without really knowing what they are because there is no valid reason to open a decoy PDF file at every startup, and adding the HIDDEN file attribute is usually reserved for executables and DLLs.",T1564.001,Hidden Files and Directories,3,57
139,"Impair Defenses: Disable or Modify System Firewall Uses batch scripts, such as rdp.bat or SERVI.bat, to modify the firewall to allow remote administration and RDP.",T1059.003,Windows Command Shell,5,12


Semantic retrieval performs better overall, but lexical retrieval can remain advantageous when threat descriptions contain distinct terms that directly overlaps with ATT&CK documentation.

**cases where all methods succeed**

In [28]:
all_success = results[
    results["bow_top5"]
    &
    results["tfidf_top5"]
    &
    results["embedding_top5"]
]

In [29]:
print(
    "Top-5 successes for all methods:",
    len(all_success)
)

Top-5 successes for all methods: 478


In [30]:
all_success[
    [
        "sentence",
        "technique_name",
        "bow_rank",
        "tfidf_rank",
        "embedding_rank"
    ]
].head(5)

,sentence,technique_name,bow_rank,tfidf_rank,embedding_rank
4,"The malware is a DLL that is launched using rundll32.exe: “C:\Windows\perfc.dat”,#1 18",Rundll32,1,1,2
8,"It calls OpenProcess on lsass.exe with access flag set to VM_READ, and looks for the modules wdigest.dll and lsasrv.dll loaded in the lsass.exe process.",LSASS Memory,1,1,1
9,"Server service, which both enforces security policies and acts as the security package manager for the LSA This file extracts credentials from LSASS similar to Mimikatz.",LSASS Memory,2,2,2
14,The following command will execute on the remote as the specified user: C:\Windows\system32\wbem\wmic.exe /node:”<server name>” /user:”<username>”,Windows Management Instrumentation,3,1,2
27,The <HH:MM> value depends on the second parameter when being invoked by rundll32.exe.,Rundll32,1,1,2


**cases where all fail**

In [31]:
all_fail = results[
    (~results["bow_top5"])
    &
    (~results["tfidf_top5"])
    &
    (~results["embedding_top5"])
]

In [32]:
print(
    "Top-5 failures for all methods:",
    len(all_fail)
)

Top-5 failures for all methods: 1598


In [33]:
all_fail[
    [
        "sentence",
        "technique_id",
        "technique_name",
        "bow_rank",
        "tfidf_rank",
        "embedding_rank"
    ]
].head(10)

,sentence,technique_id,technique_name,bow_rank,tfidf_rank,embedding_rank
0,"It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.",T1210,Exploitation of Remote Services,29,10,24
1,"Once a machine is infected by NotPetya, a series of malicious activities ensue, including the following: Dropped files Process hashes and process privilege checks Credential theft Token impersonation Malware propagation Network node enumeration SMB copy and remote execution SMBv1 exploitation via EternalBlue UNC write malware to admin$ on remote target Remote execution of the malware MBR ransomware Physical drive manipulation MFT encryption File encryption System shutdown Anti-forensics Ransomware instructions for file recovery occur after the infection process has completed.",T1570,Lateral Tool Transfer,52,31,20
2,Ransomware DLL C:\windows\perfc.dat,T1140,Deobfuscate/Decode Files or Information,122,122,27
3,"The malware decompresses its resource named 0x3 of type RT_RCDATA, and writes the contents to C:\Windows\dllhost.dat.",T1140,Deobfuscate/Decode Files or Information,42,22,14
5,It is executed with the following arguments: #1 → This is the ordinal number of the exported function 18 → Minutes used to determine how long to wait for the scheduled shutdown “username:pass” → Credentials to be used to propagate the malware on the network.,T1057,Process Discovery,123,202,18
6,The credential theft module is dropped if the malware has one of the following privileges: (It should be noted that the PROC_FLAG can any of the possible values) SeDebugPrivilege SeShutdownPrivilege & SeDebugPrivilege SeDebugPrivilege & SeTcbPrivilege SeShutdownPrivilege & SeDebugPrivilege & SeTcbPrivilege The MBR overwrite and MFT encryption subroutine is not executed if avp.exe is found running on the system The EternalBlue subroutine is not executed if either ns.exe or ccSvcHst.exe are found running on the system The system is rebooted using the API NtRaiseHardError if the process has at least the SeShutdownPrivilege level Credential Theft Credentials are provided to the malware in two ways:,T1106,Native API,93,89,212
7,"The malware decompresses a resource (0x1 if the OS is x86, 0x2 if the OS is x64) of type RT_RCDATA using zlib 1.2.8 compression.",T1140,Deobfuscate/Decode Files or Information,17,28,16
10,The impersonated token replaces the current thread token with SetThreadToken and the thread is resumed.,T1570,Lateral Tool Transfer,532,532,497
15,The malware takes the following steps to modify the MBR: Reads the MBR and encodes it using XOR encoding with key 0x7 The first sector is overwritten by a custom boot loader,T1027,Obfuscated Files or Information,293,82,226
16,"In addition, another random array of bytes 0x8 in length is created using the same API call; this is used as the nonce for the encryption.",T1106,Native API,6,7,82


After inspecting all failures, here are some reasons why it looked like it failed:

- too many behaviors in one sentence
- short/poor queries such as "Ransomware DLL C:\windows\perfc.dat"
- very lengthy and complex queries such as #6
- technology description mismatch. The query and our corpus may describe same behavior differently

**Which techniques are hardest?**

In [34]:
technique_performance = (
    results
    .groupby(
        ["technique_id", "technique_name"]
    )
    .agg(
        examples=("sentence", "count"),
        bow_mean_rank=("bow_rank", "mean"),
        tfidf_mean_rank=("tfidf_rank", "mean"),
        embedding_mean_rank=("embedding_rank", "mean")
    )
    .reset_index()
)

In [36]:
#techniques embedding struggle with
technique_performance.sort_values(
    "embedding_mean_rank",
    ascending=False
).head(10)

,technique_id,technique_name,examples,bow_mean_rank,tfidf_mean_rank,embedding_mean_rank
18,T1072,Software Deployment Tools,9,56.444444,63.000000,159.333333
19,T1074.001,Local Data Staging,20,103.750000,99.600000,147.400000
35,T1219,Remote Access Tools,46,82.413043,87.282609,118.630435
26,T1106,Native API,128,107.429688,89.976562,98.726562
1,T1005,Data from Local System,29,99.827586,96.862069,88.241379
37,T1518.001,Security Software Discovery,22,137.636364,126.136364,85.000000
46,T1573.001,Symmetric Cryptography,37,96.054054,78.702703,83.783784
21,T1082,System Information Discovery,44,90.159091,88.090909,78.318182
20,T1078,Valid Accounts,102,91.519608,87.117647,76.774510
31,T1190,Exploit Public-Facing Application,50,70.980000,48.120000,76.020000


In [38]:
#easier for embedding
technique_performance.sort_values(
    "embedding_mean_rank"
).head(10)

,technique_id,technique_name,examples,bow_mean_rank,tfidf_mean_rank,embedding_mean_rank
34,T1218.011,Rundll32,28,4.785714,2.035714,4.571429
15,T1068,Exploitation for Privilege Escalation,6,15.333333,5.500000,10.000000
13,T1057,Process Discovery,38,18.236842,15.789474,19.473684
16,T1070.004,File Deletion,65,174.400000,155.553846,19.630769
36,T1484.001,Group Policy Modification,14,9.142857,6.571429,19.785714
9,T1047,Windows Management Instrumentation,48,25.125000,17.062500,20.187500
6,T1033,System Owner/User Discovery,12,91.750000,80.416667,23.833333
32,T1204.002,Malicious File,65,46.338462,43.030769,24.415385
0,T1003.001,LSASS Memory,74,39.716216,21.702703,25.972973
2,T1012,Query Registry,13,43.384615,38.615385,26.076923


In [39]:
#top 5 recall
technique_top5 = (
    results
    .groupby(
        ["technique_id", "technique_name"]
    )
    .agg(
        examples=("sentence", "count"),

        bow_top5=(
            "bow_top5",
            "mean"
        ),

        tfidf_top5=(
            "tfidf_top5",
            "mean"
        ),

        embedding_top5=(
            "embedding_top5",
            "mean"
        )
    )
    .reset_index()
)

In [41]:
technique_top5.sort_values(
    "embedding_top5"
).head(10)

,technique_id,technique_name,examples,bow_top5,tfidf_top5,embedding_top5
20,T1078,Valid Accounts,102,0.186275,0.147059,0.078431
7,T1036.005,Match Legitimate Resource Name or Location,50,0.200000,0.120000,0.120000
26,T1106,Native API,128,0.273438,0.257812,0.125000
1,T1005,Data from Local System,29,0.172414,0.206897,0.137931
5,T1027,Obfuscated Files or Information,467,0.042827,0.064240,0.171306
31,T1190,Exploit Public-Facing Application,50,0.140000,0.300000,0.180000
37,T1518.001,Security Software Discovery,22,0.318182,0.363636,0.181818
21,T1082,System Information Discovery,44,0.227273,0.227273,0.181818
24,T1095,Non-Application Layer Protocol,43,0.093023,0.209302,0.186047
19,T1074.001,Local Data Staging,20,0.450000,0.350000,0.200000


**does query length matter?**

In [42]:
results["word_count"] = (
    results["sentence"]
    .str.split()
    .str.len()
)

In [43]:
results["length_group"] = pd.cut(
    results["word_count"],
    bins=[
        0,
        10,
        20,
        40,
        float("inf")
    ],
    labels=[
        "1-10 words",
        "11-20 words",
        "21-40 words",
        "41+ words"
    ]
)

In [44]:
length_performance = (
    results
    .groupby(
        "length_group",
        observed=True
    )
    .agg(
        examples=("sentence", "count"),
        bow_top5=("bow_top5", "mean"),
        tfidf_top5=("tfidf_top5", "mean"),
        embedding_top5=("embedding_top5", "mean")
    )
)

length_performance

,examples,bow_top5,tfidf_top5,embedding_top5
length_group,,,,
1-10 words,424,0.320755,0.389151,0.441038
11-20 words,1227,0.277099,0.338223,0.421353
21-40 words,1207,0.202983,0.257664,0.294946
41+ words,219,0.141553,0.210046,0.210046


In [45]:
#summary
summary = pd.DataFrame({
    "Category": [
        "All methods Top-5 success",
        "Embedding success / TF-IDF failure",
        "TF-IDF success / Embedding failure",
        "All methods Top-5 failure"
    ],

    "Count": [
        len(all_success),
        len(embedding_wins),
        len(tfidf_wins),
        len(all_fail)
    ]
})

summary

,Category,Count
0,All methods Top-5 success,478
1,Embedding success / TF-IDF failure,474
2,TF-IDF success / Embedding failure,305
3,All methods Top-5 failure,1598


In [46]:
#save
results.to_csv(
    "retrieval_error_analysis.csv",
    index=False
)

In [47]:
#save
technique_top5.to_csv(
    "technique_retrieval_performance.csv",
    index=False
)

**Result**

All methods Top-5 success: 478
Embedding Top-5 success / TF-IDF failure: 474
TF-IDF Top-5 success / Embedding failure: 305
All methods Top-5 failure: 1598

------------------------------------------------------------------------------------------------
After inspecting all failures, here are some reasons why it looked like it failed:

- too many behaviors in one sentence
- short/poor queries such as "Ransomware DLL C:\windows\perfc.dat"
- very lengthy and complex queries such as #6
- technology description mismatch. The query and our corpus may describe same behavior differently